# Prompt Engineering Best Practices: Textual Inference & Sentiment Analysis

[Article](https://todatabeyond.com/blog/prompt-engineering-best-practices-textual-inference-and-sentiment-analysis). [Series contents](./README.md).

Install `pip install openai==2.43.0 python-dotenv==1.2.2`. Set OPENAI_API_KEY in the environment or an uncommitted local .env. MODEL defaults to gpt-5.4-mini. Calls incur charges; never put a key in cells. Original code and output remain in the nonexecuting appendix. Examples keep the complete source input texts and loops; outputs can change between invocations.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion(prompt, model=MODEL):
    result = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        reasoning_effort="none", max_completion_tokens=2048)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def parse_json_object(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    raise ValueError("No JSON object found")
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
lamp_review = """
Needed a nice lamp for my bedroom, and this one had  additional storage and not too high of a price point.  Got it fast.  The string to our lamp broke during the  transit and the company happily sent over a new one.  Came within a few days as well. It was easy to put  together.  I had a missing part, so I contacted their  support and they very quickly got me the missing piece!  Lumina seems to me to be a great company that cares  about their customers and products!!
"""

In [ ]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
sentiment_response = response

In [ ]:
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive"  or "negative".

Review text: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
concise_sentiment = response

In [ ]:
prompt = f"""
Identify a list of emotions that the writer of the  following review is expressing. Include no more than  five items in the list. Format your answer as a list of  lower-case words separated by commas.

Review text: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
emotions_response = response

In [ ]:
prompt = f"""
Is the writer of the following review expressing anger? The review is delimited with triple backticks.  Give your answer as either yes or no.

Review text: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
anger_response = response

In [ ]:
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks.  Format your response as a JSON object with  "Item" and "Brand" as the keys. 
If the information isn't present, use "unknown"  as the value.
Make your response as short as possible.
  
Review text: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
item_brand = parse_json_object(response)

In [ ]:
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks.  Format your response as a JSON object with  "Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown"  as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: ```{lamp_review}```
"""
response = get_completion(prompt)
print(response)
combined = parse_json_object(response)

In [ ]:
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""

In [ ]:
prompt = f"""
Determine five topics that are being discussed in the  following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: ```{story}```
"""
prompt += "\nReturn exactly five comma-separated labels, each one or two whitespace-separated words. Abbreviate longer organization names (for example SSA). No numbering or explanation."
response = get_completion(prompt)
print(response)
topics_response = response

In [ ]:
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]

In [ ]:
prompt = f"""
Determine whether each item in the following list of  topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as list with 0 or 1 for each topic. 
List of topics: {", ".join(topic_list)}

Text sample: ```{story}```
"""
prompt += "\nReturn only a JSON array of exactly five integer flags (0 or 1), in the listed order. No code fence or explanation."
response = get_completion(prompt)
print(response)
def parse_topic_presence(text, topics):
    flags = json.loads(text)
    if not isinstance(flags, list) or len(flags) != len(topics):
        raise ValueError("One flag per topic is required")
    if any(type(v) is not int or v not in (0, 1) for v in flags):
        raise ValueError("Flags must be integer 0 or 1")
    return flags
def alert_for(mapping):
    return "ALERT: New NASA story!" if mapping.get("nasa") == 1 else "No NASA alert."
presence = parse_topic_presence(response, topic_list)
topic_presence = dict(zip(topic_list, presence, strict=True))
print(json.dumps(topic_presence, indent=2))
print(alert_for(topic_presence))

In [ ]:
assert "positive" in sentiment_response.lower()
assert concise_sentiment.strip().lower().rstrip(".") == "positive"
emotions = [e.strip() for e in emotions_response.split(",")]
assert 1 <= len(emotions) <= 5 and all(e == e.lower() and e for e in emotions)
assert anger_response.strip().lower().rstrip(".") == "no"
assert set(item_brand) == {"Item", "Brand"}
assert item_brand["Item"].lower() == "lamp" and item_brand["Brand"] == "Lumina"
assert set(combined) == {"Sentiment", "Anger", "Item", "Brand"}
assert combined["Sentiment"].lower() == "positive" and combined["Anger"] is False
assert combined["Item"].lower() == "lamp" and combined["Brand"] == "Lumina"
topics = [x.strip() for x in topics_response.split(",")]
assert len(topics) == 5 and all(1 <= len(t.split()) <= 2 for t in topics)
assert "nasa" in topics_response.lower()
assert presence[0] == 1 and presence[3] == 1
assert alert_for(topic_presence) == "ALERT: New NASA story!"
assert alert_for(dict.fromkeys(topic_list, 0)) == "No NASA alert."
for bad in ["[1, 0]", "[1, 0, 0, 1, true]", "[1, 0, 2, 1, 1]", "not json"]:
    try: parse_topic_presence(bad, topic_list)
    except (ValueError, json.JSONDecodeError): pass
    else: raise AssertionError("Invalid topic flags accepted")
print("PASS: positive sentiment, anger=false, both complete extraction schemas, five short topics, known NASA/employee topics, both alert branches and invalid flag rejection.")

## Original archive appendix — not executed

Historical SDK fragments, examples and transcripts below are retained verbatim, not used as fresh test results.

### Original pre block 1

```text
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')
```

### Original pre block 2

```text
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

### Original pre block 3

```text
lamp_review = """
Needed a nice lamp for my bedroom, and this one had \
additional storage and not too high of a price point. \
Got it fast.  The string to our lamp broke during the \
transit and the company happily sent over a new one. \
Came within a few days as well. It was easy to put \
together.  I had a missing part, so I contacted their \
support and they very quickly got me the missing piece! \
Lumina seems to me to be a great company that cares \
about their customers and products!!
"""
```

### Original pre block 4

```text
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 5

```text
prompt = f"""
What is the sentiment of the following product review, 
which is delimited with triple backticks?

Give your answer as a single word, either "positive" \
or "negative".

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 6

```text
prompt = f"""
Identify a list of emotions that the writer of the \
following review is expressing. Include no more than \
five items in the list. Format your answer as a list of \
lower-case words separated by commas.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 7

```text
prompt = f"""
Is the writer of the following review expressing anger?\
The review is delimited with triple backticks. \
Give your answer as either yes or no.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 8

```text
prompt = f"""
Identify the following items from the review text: 
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Item" and "Brand" as the keys. 
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
  
Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 9

```text
prompt = f"""
Identify the following items from the review text: 
- Sentiment (positive or negative)
- Is the reviewer expressing anger? (true or false)
- Item purchased by reviewer
- Company that made the item

The review is delimited with triple backticks. \
Format your response as a JSON object with \
"Sentiment", "Anger", "Item" and "Brand" as the keys.
If the information isn't present, use "unknown" \
as the value.
Make your response as short as possible.
Format the Anger value as a boolean.

Review text: '''{lamp_review}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 10

```text
story = """
In a recent survey conducted by the government, 
public sector employees were asked to rate their level 
of satisfaction with the department they work at. 
The results revealed that NASA was the most popular 
department with a satisfaction rating of 95%.

One NASA employee, John Smith, commented on the findings, 
stating, "I'm not surprised that NASA came out on top. 
It's a great place to work with amazing people and 
incredible opportunities. I'm proud to be a part of 
such an innovative organization."

The results were also welcomed by NASA's management team, 
with Director Tom Johnson stating, "We are thrilled to 
hear that our employees are satisfied with their work at NASA. 
We have a talented and dedicated team who work tirelessly 
to achieve our goals, and it's fantastic to see that their 
hard work is paying off."

The survey also revealed that the 
Social Security Administration had the lowest satisfaction 
rating, with only 45% of employees indicating they were 
satisfied with their job. The government has pledged to 
address the concerns raised by employees in the survey and 
work towards improving job satisfaction across all departments.
"""
```

### Original pre block 11

```text
prompt = f"""
Determine five topics that are being discussed in the \
following text, which is delimited by triple backticks.

Make each item one or two words long. 

Format your response as a list of items separated by commas.

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 12

```text
topic_list = [
    "nasa", "local government", "engineering", 
    "employee satisfaction", "federal government"
]
```

### Original pre block 13

```text
prompt = f"""
Determine whether each item in the following list of \
topics is a topic in the text below, which
is delimited with triple backticks.

Give your answer as list with 0 or 1 for each topic.\

List of topics: {", ".join(topic_list)}

Text sample: '''{story}'''
"""
response = get_completion(prompt)
print(response)
```

### Original transcript 1

```text
The sentiment of the product review is positive.
```

### Original transcript 2

```text
Postive
```

### Original transcript 3

```text
satisfied, grateful, impressed, pleased, happy
```

### Original transcript 4

```text
No
```

### Original transcript 5

```text
{
 “Item”: “lamp”,
 “Brand”: “Lumina”
}
```

### Original transcript 6

```text
{
 “Sentiment”: “positive”,
 “Anger”: false,
 “Item”: “lamp”,
 “Brand”: “Lumina”
}
```

### Original transcript 7

```text
1. Government survey
2. Department satisfaction rating
3. NASA
4. Social Security Administration
5. Job satisfaction improvement
```

### Original transcript 8

```text
[1, 0, 0, 1, 1]
```